# Optuna Tutorial

A compact walkthrough of hyperparameter search patterns using the Iris dataset.

**What you will see**
- Baseline training without Optuna
- A simple Optuna study
- Conditional search spaces
- Basic study visualizations

In [1]:
import optuna

optuna.__version__

'4.5.0'

## Baseline (No Optuna)

A fixed-parameter RandomForest is trained and evaluated with cross-validation.

In [2]:
import sklearn.datasets
import sklearn.ensemble
import sklearn.model_selection


def objective():
    iris = sklearn.datasets.load_iris()  # Prepare the data.

    clf = sklearn.ensemble.RandomForestClassifier(n_estimators=5, max_depth=3)  # Define the model.

    return sklearn.model_selection.cross_val_score(
        clf, iris.data, iris.target, n_jobs=-1, cv=3
    ).mean()  # Train and evaluate the model.


print("Accuracy: {}".format(objective()))

Accuracy: 0.9466666666666667


## Optuna: Simple Study

Search over `n_estimators` and `max_depth` and report the best trial.

In [3]:
def objective(trial):
    iris = sklearn.datasets.load_iris()

    n_estimators = trial.suggest_int("n_estimators", 2, 20)
    max_depth = int(trial.suggest_float("max_depth", 1, 32, log=True))

    clf = sklearn.ensemble.RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)

    return sklearn.model_selection.cross_val_score(
        clf, iris.data, iris.target, n_jobs=-1, cv=3
    ).mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

trial = study.best_trial

print("Accuracy: {}".format(trial.value))
print("Best hyperparameters: {}".format(trial.params))

[I 2026-02-20 12:54:58,270] A new study created in memory with name: no-name-00fd51d2-a2ee-4217-afbf-5c256161eccf
[I 2026-02-20 12:54:59,117] Trial 0 finished with value: 0.96 and parameters: {'n_estimators': 18, 'max_depth': 7.968464988949526}. Best is trial 0 with value: 0.96.
[I 2026-02-20 12:54:59,971] Trial 1 finished with value: 0.9666666666666667 and parameters: {'n_estimators': 10, 'max_depth': 12.295949241987813}. Best is trial 1 with value: 0.9666666666666667.
[I 2026-02-20 12:55:00,816] Trial 2 finished with value: 0.96 and parameters: {'n_estimators': 16, 'max_depth': 16.795035761972347}. Best is trial 1 with value: 0.9666666666666667.
[I 2026-02-20 12:55:00,841] Trial 3 finished with value: 0.94 and parameters: {'n_estimators': 9, 'max_depth': 1.3692964583363831}. Best is trial 1 with value: 0.9666666666666667.
[I 2026-02-20 12:55:00,865] Trial 4 finished with value: 0.96 and parameters: {'n_estimators': 13, 'max_depth': 2.397199923793284}. Best is trial 1 with value: 0.96

Accuracy: 0.9733333333333333
Best hyperparameters: {'n_estimators': 13, 'max_depth': 14.028737280924002}


## Conditional Search Space

Use `suggest_categorical` to branch between model families and tune their specific hyperparameters.

In [4]:
import sklearn.svm


def objective(trial):
    iris = sklearn.datasets.load_iris()

    classifier = trial.suggest_categorical("classifier", ["RandomForest", "SVC"])

    if classifier == "RandomForest":
        n_estimators = trial.suggest_int("n_estimators", 2, 20)
        max_depth = int(trial.suggest_float("max_depth", 1, 32, log=True))

        clf = sklearn.ensemble.RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth
        )
    else:
        c = trial.suggest_float("svc_c", 1e-10, 1e10, log=True)

        clf = sklearn.svm.SVC(C=c, gamma="auto")

    return sklearn.model_selection.cross_val_score(
        clf, iris.data, iris.target, n_jobs=-1, cv=3
    ).mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

trial = study.best_trial

print("Accuracy: {}".format(trial.value))
print("Best hyperparameters: {}".format(trial.params))

[I 2026-02-20 12:55:27,779] A new study created in memory with name: no-name-2b482e37-7e94-4580-8dd1-16f03d03b3b1
[I 2026-02-20 12:55:27,818] Trial 0 finished with value: 0.9466666666666667 and parameters: {'classifier': 'RandomForest', 'n_estimators': 20, 'max_depth': 14.70510297449332}. Best is trial 0 with value: 0.9466666666666667.
[I 2026-02-20 12:55:27,832] Trial 1 finished with value: 0.9666666666666667 and parameters: {'classifier': 'RandomForest', 'n_estimators': 7, 'max_depth': 4.6192054175252215}. Best is trial 1 with value: 0.9666666666666667.
[I 2026-02-20 12:55:27,856] Trial 2 finished with value: 0.96 and parameters: {'classifier': 'RandomForest', 'n_estimators': 20, 'max_depth': 3.098765101525833}. Best is trial 1 with value: 0.9666666666666667.
[I 2026-02-20 12:55:27,870] Trial 3 finished with value: 0.7600000000000001 and parameters: {'classifier': 'RandomForest', 'n_estimators': 8, 'max_depth': 1.6565241662474626}. Best is trial 1 with value: 0.9666666666666667.
[I 2

Accuracy: 0.9866666666666667
Best hyperparameters: {'classifier': 'SVC', 'svc_c': 4.414551936372888}


## Visualizations

Quick diagnostics for optimization behavior and parameter interactions.

In [5]:
optuna.visualization.plot_optimization_history(study)

In [6]:
optuna.visualization.plot_slice(study)

In [7]:
optuna.visualization.plot_contour(study, params=["n_estimators", "max_depth"])